# PlagNet SimCLR embedding analysis

Analysis steps included below:

1. Embedding variance and collapse checks  
2. Nearest-neighbour crystal inspection  
3. PCA  
4. UMAP visualisation  
5. Clustering across several values of k  
6. Cluster stability across seeds and data perturbations  
7. Association with sample identity  
8. Confounder tests for size, shape, brightness, background, and padding  
9. Aligned comparison with the legacy SimSiam embeddings  

## 0. Environment

Run this once if the analysis environment is missing packages:

```python
%pip install numpy pandas pyarrow pillow matplotlib scipy scikit-learn umap-learn tqdm
```

Restart the notebook kernel after installation if requested.


In [ ]:
from __future__ import annotations

import json
import math
import warnings
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from IPython.display import display
from PIL import Image, ImageDraw
from scipy.stats import chi2_contingency, spearmanr
from tqdm.auto import tqdm

from sklearn.cluster import KMeans
from sklearn.decomposition import PCA
from sklearn.linear_model import LogisticRegression, Ridge
from sklearn.metrics import (
    adjusted_mutual_info_score,
    adjusted_rand_score,
    balanced_accuracy_score,
    calinski_harabasz_score,
    completeness_score,
    davies_bouldin_score,
    homogeneity_score,
    normalized_mutual_info_score,
    silhouette_score,
)
from sklearn.model_selection import (
    KFold,
    StratifiedKFold,
    cross_val_score,
)
from sklearn.neighbors import NearestNeighbors
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import Normalizer, StandardScaler

import umap

warnings.filterwarnings("ignore", category=FutureWarning)

print("Imports completed.")


## 0.1 paths and analysis settings

Change SIMCLR_RUN only if your local directory differs.

The legacy SimSiam path should be added later in Step 9 for comparison


In [ ]:
PROJECT_ROOT = Path("/home/ars223/MSc-GEMS/IRP26")

SIMCLR_RUN = (
    PROJECT_ROOT
    / "hpc_results"
    / "simclr"
    / "full_l40s_seed42_3134848"
)

DATASET_ROOT = PROJECT_ROOT / "extracted_plags"

EMBEDDING_FILE = (
    SIMCLR_RUN
    / "embeddings"
    / "simclr_backbone_embeddings_safe.npz"
)

# Fall back to the original verified file if the safe Unicode version
# has not yet been created.
if not EMBEDDING_FILE.exists():
    EMBEDDING_FILE = (
        SIMCLR_RUN
        / "embeddings"
        / "simclr_backbone_embeddings.npz"
    )

EMBEDDING_MANIFEST_FILE = (
    SIMCLR_RUN
    / "embeddings"
    / "embedding_manifest.csv"
)

ANALYSIS_ROOT = SIMCLR_RUN / "analysis"
TABLES_ROOT = ANALYSIS_ROOT / "tables"
FIGURES_ROOT = ANALYSIS_ROOT / "figures"
NEIGHBOURS_ROOT = ANALYSIS_ROOT / "nearest_neighbours"

for directory in [
    ANALYSIS_ROOT,
    TABLES_ROOT,
    FIGURES_ROOT,
    NEIGHBOURS_ROOT,
]:
    directory.mkdir(parents=True, exist_ok=True)

SEED = 42
PCA_COMPONENTS = 50
NEIGHBOUR_COUNT = 10
K_VALUES = list(range(2, 13))
STABILITY_RUNS = 10
SILHOUETTE_SAMPLE_SIZE = 4_000
UMAP_NEIGHBOURS = 30
UMAP_MIN_DIST = 0.10

# This is initially selected using maximum silhouette score.
# After reviewing all metrics and stability, you may manually replace it.
SELECTED_K = None

# Add the legacy PlagNet output when available, for example:
# SIMSIAM_NPZ = PROJECT_ROOT / "legacy_simsiam/SSL_output_20260408_135606.npz"
SIMSIAM_NPZ = None

print("SimCLR run:", SIMCLR_RUN)
print("Embedding file:", EMBEDDING_FILE)
print("Dataset root:", DATASET_ROOT)
print("Analysis output:", ANALYSIS_ROOT)


## 0.2 load and align the SimCLR embeddings

The original .npz contains trusted text arrays saved by the training job, therefore the loader permits pickle only when necessary. Numerical and path alignment are rechecked here.


In [ ]:
if not EMBEDDING_FILE.exists():
    raise FileNotFoundError(EMBEDDING_FILE)

if not EMBEDDING_MANIFEST_FILE.exists():
    raise FileNotFoundError(EMBEDDING_MANIFEST_FILE)

manifest = pd.read_csv(EMBEDDING_MANIFEST_FILE)

try:
    saved = np.load(EMBEDDING_FILE, allow_pickle=False)
    _ = saved["relative_path"]
except ValueError:
    saved = np.load(EMBEDDING_FILE, allow_pickle=True)

with saved:
    embeddings_raw = np.asarray(saved["embeddings"], dtype=np.float32)
    embeddings_l2 = np.asarray(saved["embeddings_l2"], dtype=np.float32)
    saved_paths = np.asarray(saved["relative_path"], dtype=str)
    saved_ids = np.asarray(saved["crystal_id"], dtype=str)
    saved_samples = np.asarray(saved["sample"], dtype=str)

manifest_paths = manifest["relative_path"].astype(str).to_numpy()
manifest_ids = manifest["crystal_id"].astype(str).to_numpy()
manifest_samples = manifest["sample"].astype(str).to_numpy()

assert embeddings_raw.shape == (12_344, 512)
assert embeddings_l2.shape == (12_344, 512)
assert len(manifest) == 12_344
assert np.isfinite(embeddings_raw).all()
assert np.isfinite(embeddings_l2).all()
assert np.array_equal(saved_paths, manifest_paths)
assert np.array_equal(saved_ids, manifest_ids)
assert np.array_equal(saved_samples, manifest_samples)

manifest["absolute_path"] = manifest["relative_path"].map(
    lambda value: DATASET_ROOT / str(value)
)

missing_images = manifest.loc[
    ~manifest["absolute_path"].map(Path.exists),
    "absolute_path",
]

if not missing_images.empty:
    raise FileNotFoundError(
        f"{len(missing_images)} source images are missing. "
        f"First example: {missing_images.iloc[0]}"
    )

print("Embedding shape:", embeddings_raw.shape)
print("Manifest rows:", len(manifest))
print("Samples:", sorted(manifest["sample"].unique()))
print("All input checks passed.")


# 1. Embedding variance and collapse checks

A fully collapsed representation would give nearly identical vectors for all crystals. We check:

- feature standard deviations;
- embedding norm distribution;
- average cosine similarity between random pairs;
- covariance eigenvalues;
- effective rank.

Effective rank near 1 would be severe collapse. A much larger value indicates that many independent directions are being used, although there is no universal pass/fail threshold.


In [ ]:
def effective_rank_from_embeddings(values: np.ndarray) -> tuple[float, np.ndarray]:
    centred = values - values.mean(axis=0, keepdims=True)
    covariance = np.cov(centred, rowvar=False)
    eigenvalues = np.linalg.eigvalsh(covariance)
    eigenvalues = np.clip(eigenvalues, 0, None)
    eigenvalues = eigenvalues[::-1]

    probabilities = eigenvalues / np.clip(eigenvalues.sum(), 1e-12, None)
    positive = probabilities > 0
    entropy = -np.sum(probabilities[positive] * np.log(probabilities[positive]))
    return float(np.exp(entropy)), eigenvalues


rng = np.random.default_rng(SEED)

feature_std = embeddings_raw.std(axis=0)
raw_norms = np.linalg.norm(embeddings_raw, axis=1)
l2_norms = np.linalg.norm(embeddings_l2, axis=1)

pair_count = 50_000
left_indices = rng.integers(0, len(embeddings_l2), size=pair_count)
right_indices = rng.integers(0, len(embeddings_l2), size=pair_count)

different = left_indices != right_indices
random_pair_cosines = np.sum(
    embeddings_l2[left_indices[different]]
    * embeddings_l2[right_indices[different]],
    axis=1,
)

effective_rank, covariance_eigenvalues = effective_rank_from_embeddings(
    embeddings_raw
)

collapse_summary = pd.DataFrame(
    {
        "metric": [
            "number_of_images",
            "embedding_dimensions",
            "near_zero_std_dimensions",
            "mean_feature_std",
            "median_feature_std",
            "minimum_feature_std",
            "maximum_feature_std",
            "mean_raw_embedding_norm",
            "std_raw_embedding_norm",
            "mean_random_pair_cosine",
            "std_random_pair_cosine",
            "effective_rank",
        ],
        "value": [
            len(embeddings_raw),
            embeddings_raw.shape[1],
            int((feature_std < 1e-8).sum()),
            float(feature_std.mean()),
            float(np.median(feature_std)),
            float(feature_std.min()),
            float(feature_std.max()),
            float(raw_norms.mean()),
            float(raw_norms.std()),
            float(random_pair_cosines.mean()),
            float(random_pair_cosines.std()),
            effective_rank,
        ],
    }
)

collapse_summary.to_csv(
    TABLES_ROOT / "01_embedding_collapse_summary.csv",
    index=False,
)

display(collapse_summary)


In [ ]:
plt.figure(figsize=(8, 5))
plt.hist(feature_std, bins=40)
plt.xlabel("Feature standard deviation")
plt.ylabel("Number of embedding dimensions")
plt.title("SimCLR feature variance")
plt.tight_layout()
plt.savefig(
    FIGURES_ROOT / "01_feature_standard_deviation.png",
    dpi=200,
    bbox_inches="tight",
)
plt.show()


In [ ]:
explained = covariance_eigenvalues / np.clip(
    covariance_eigenvalues.sum(),
    1e-12,
    None,
)
cumulative_explained = np.cumsum(explained)

plt.figure(figsize=(8, 5))
plt.plot(
    np.arange(1, len(cumulative_explained) + 1),
    cumulative_explained,
)
plt.xlabel("Number of covariance eigenvectors")
plt.ylabel("Cumulative variance explained")
plt.title("SimCLR embedding dimensionality")
plt.grid(alpha=0.3)
plt.tight_layout()
plt.savefig(
    FIGURES_ROOT / "01_cumulative_embedding_variance.png",
    dpi=200,
    bbox_inches="tight",
)
plt.show()


# 2. Nearest-neighbour crystal inspection

Nearest neighbours are calculated in the L2-normalised 512-dimensional embedding space using cosine distance.

One query is selected reproducibly from each sample. The contact sheets allow inspection on whether neighbours share zoning, morphology, colour, size, or merely background characteristics.


In [ ]:
nearest_neighbour_model = NearestNeighbors(
    n_neighbors=NEIGHBOUR_COUNT + 1,
    metric="cosine",
    n_jobs=-1,
)

nearest_neighbour_model.fit(embeddings_l2)

all_distances, all_indices = nearest_neighbour_model.kneighbors(
    embeddings_l2
)

# Remove each image's self-match in column zero.
neighbour_distances = all_distances[:, 1:]
neighbour_indices = all_indices[:, 1:]

np.savez_compressed(
    TABLES_ROOT / "02_nearest_neighbour_indices.npz",
    indices=neighbour_indices.astype(np.int32),
    cosine_distances=neighbour_distances.astype(np.float32),
)


In [ ]:
def pad_and_resize(
    image: Image.Image,
    size: int = 180,
) -> Image.Image:
    image = image.convert("RGB")
    width, height = image.size
    side = max(width, height)

    canvas = Image.new("RGB", (side, side), (0, 0, 0))
    canvas.paste(
        image,
        ((side - width) // 2, (side - height) // 2),
    )
    return canvas.resize((size, size))


def make_neighbour_contact_sheet(
    query_index: int,
    neighbours: np.ndarray,
    distances: np.ndarray,
    tile_size: int = 180,
    label_height: int = 38,
) -> Image.Image:
    shown_indices = [query_index] + neighbours.tolist()
    shown_distances = [0.0] + distances.tolist()

    sheet_width = tile_size * len(shown_indices)
    sheet_height = tile_size + label_height

    sheet = Image.new(
        "RGB",
        (sheet_width, sheet_height),
        (255, 255, 255),
    )
    draw = ImageDraw.Draw(sheet)

    for position, (index, distance) in enumerate(
        zip(shown_indices, shown_distances)
    ):
        row = manifest.iloc[index]
        image = pad_and_resize(
            Image.open(row["absolute_path"]),
            size=tile_size,
        )

        left = position * tile_size
        sheet.paste(image, (left, 0))

        if position == 0:
            label = f"QUERY\n{row['sample']}"
        else:
            label = f"N{position} d={distance:.3f}\n{row['sample']}"

        draw.multiline_text(
            (left + 4, tile_size + 2),
            label,
            fill=(0, 0, 0),
            spacing=1,
        )

    return sheet


query_indices = []

for sample, group in manifest.groupby("sample", sort=True):
    query_indices.append(
        int(
            group.sample(
                n=1,
                random_state=SEED,
            ).index[0]
        )
    )

neighbour_rows = []

for query_index in query_indices:
    query_row = manifest.iloc[query_index]
    neighbours = neighbour_indices[query_index]
    distances = neighbour_distances[query_index]

    contact_sheet = make_neighbour_contact_sheet(
        query_index,
        neighbours,
        distances,
    )

    output_path = (
        NEIGHBOURS_ROOT
        / f"{query_row['sample']}_{query_row['crystal_id']}.png"
    )
    contact_sheet.save(output_path)

    display(contact_sheet)

    for rank, (neighbour_index, distance) in enumerate(
        zip(neighbours, distances),
        start=1,
    ):
        neighbour_row = manifest.iloc[int(neighbour_index)]

        neighbour_rows.append(
            {
                "query_index": query_index,
                "query_crystal_id": query_row["crystal_id"],
                "query_sample": query_row["sample"],
                "neighbour_rank": rank,
                "neighbour_index": int(neighbour_index),
                "neighbour_crystal_id": neighbour_row["crystal_id"],
                "neighbour_sample": neighbour_row["sample"],
                "cosine_distance": float(distance),
                "same_sample": (
                    query_row["sample"]
                    == neighbour_row["sample"]
                ),
            }
        )

nearest_neighbour_table = pd.DataFrame(neighbour_rows)

nearest_neighbour_table.to_csv(
    TABLES_ROOT / "02_selected_nearest_neighbours.csv",
    index=False,
)

display(nearest_neighbour_table.head(20))


In [ ]:
same_sample_neighbour_rate = (
    manifest["sample"]
    .astype(str)
    .to_numpy()[neighbour_indices]
    == manifest["sample"].astype(str).to_numpy()[:, None]
).mean()

print(
    "Fraction of top-10 neighbours from the same sample:",
    f"{same_sample_neighbour_rate:.3f}",
)


# 3. PCA

PCA is fitted to the L2-normalised embeddings. Fifty components are retained for clustering and statistical analysis.

The two-dimensional plot is descriptive only.


In [ ]:
pca_model = PCA(
    n_components=PCA_COMPONENTS,
    svd_solver="randomized",
    random_state=SEED,
)

pca_scores = pca_model.fit_transform(embeddings_l2)

# L2-normalise the retained PCA representation before k-means.
cluster_features = Normalizer().fit_transform(pca_scores)

pca_table = manifest[
    ["crystal_id", "sample", "relative_path"]
].copy()

for component_index in range(PCA_COMPONENTS):
    pca_table[f"PC{component_index + 1}"] = pca_scores[:, component_index]

pca_table.to_parquet(
    TABLES_ROOT / "03_pca_coordinates.parquet",
    index=False,
)

pca_variance_table = pd.DataFrame(
    {
        "component": np.arange(1, PCA_COMPONENTS + 1),
        "explained_variance_ratio": pca_model.explained_variance_ratio_,
        "cumulative_explained_variance": np.cumsum(
            pca_model.explained_variance_ratio_
        ),
    }
)

pca_variance_table.to_csv(
    TABLES_ROOT / "03_pca_explained_variance.csv",
    index=False,
)

display(pca_variance_table.head(10))


In [ ]:
plt.figure(figsize=(8, 5))
plt.plot(
    pca_variance_table["component"],
    pca_variance_table["cumulative_explained_variance"],
)
plt.xlabel("Number of PCA components")
plt.ylabel("Cumulative explained variance")
plt.title("PCA explained variance")
plt.grid(alpha=0.3)
plt.tight_layout()
plt.savefig(
    FIGURES_ROOT / "03_pca_cumulative_explained_variance.png",
    dpi=200,
    bbox_inches="tight",
)
plt.show()


In [ ]:
sample_codes, sample_names = pd.factorize(
    manifest["sample"].astype(str),
    sort=True,
)

plt.figure(figsize=(9, 7))
scatter = plt.scatter(
    pca_scores[:, 0],
    pca_scores[:, 1],
    c=sample_codes,
    s=6,
    alpha=0.55,
)

handles, _ = scatter.legend_elements()
plt.legend(
    handles,
    sample_names,
    title="Sample",
    bbox_to_anchor=(1.02, 1),
    loc="upper left",
)
plt.xlabel("PC1")
plt.ylabel("PC2")
plt.title("SimCLR embeddings: PCA by sample")
plt.tight_layout()
plt.savefig(
    FIGURES_ROOT / "03_pca_by_sample.png",
    dpi=200,
    bbox_inches="tight",
)
plt.show()


# 4. UMAP visualisation

UMAP is fitted to the 50-dimensional PCA representation. It is used for visualisation, not as the space in which clusters are defined.


In [ ]:
umap_model = umap.UMAP(
    n_components=2,
    n_neighbors=UMAP_NEIGHBOURS,
    min_dist=UMAP_MIN_DIST,
    metric="euclidean",
    random_state=SEED,
)

umap_scores = umap_model.fit_transform(pca_scores)

umap_table = manifest[
    ["crystal_id", "sample", "relative_path"]
].copy()

umap_table["UMAP1"] = umap_scores[:, 0]
umap_table["UMAP2"] = umap_scores[:, 1]

umap_table.to_parquet(
    TABLES_ROOT / "04_umap_coordinates.parquet",
    index=False,
)

display(umap_table.head())


In [ ]:
plt.figure(figsize=(9, 7))
scatter = plt.scatter(
    umap_scores[:, 0],
    umap_scores[:, 1],
    c=sample_codes,
    s=6,
    alpha=0.55,
)

handles, _ = scatter.legend_elements()
plt.legend(
    handles,
    sample_names,
    title="Sample",
    bbox_to_anchor=(1.02, 1),
    loc="upper left",
)
plt.xlabel("UMAP1")
plt.ylabel("UMAP2")
plt.title("SimCLR embeddings: UMAP by sample")
plt.tight_layout()
plt.savefig(
    FIGURES_ROOT / "04_umap_by_sample.png",
    dpi=200,
    bbox_inches="tight",
)
plt.show()


# 5. clustering across several values of k

K-means is fitted in the normalised 50-component PCA space.

Metrics:

- higher silhouette is preferable;
- lower Davies–Bouldin is preferable;
- higher Calinski–Harabasz is preferable;
- inertia always decreases with k, so it should not be used alone.

The provisional SELECTED_K is the value with maximum silhouette. Geological interpretability and stability must also be considered.


In [ ]:
clustering_metric_rows = []
clusterings_by_k = {}

silhouette_sample = min(
    SILHOUETTE_SAMPLE_SIZE,
    len(cluster_features),
)

for k in tqdm(K_VALUES, desc="Evaluating k-means"):
    kmeans = KMeans(
        n_clusters=k,
        n_init=20,
        random_state=SEED,
    )

    labels = kmeans.fit_predict(cluster_features)
    clusterings_by_k[k] = labels

    clustering_metric_rows.append(
        {
            "k": k,
            "inertia": float(kmeans.inertia_),
            "silhouette": float(
                silhouette_score(
                    cluster_features,
                    labels,
                    sample_size=silhouette_sample,
                    random_state=SEED,
                )
            ),
            "davies_bouldin": float(
                davies_bouldin_score(
                    cluster_features,
                    labels,
                )
            ),
            "calinski_harabasz": float(
                calinski_harabasz_score(
                    cluster_features,
                    labels,
                )
            ),
            "smallest_cluster": int(
                np.bincount(labels).min()
            ),
            "largest_cluster": int(
                np.bincount(labels).max()
            ),
        }
    )

clustering_metrics = pd.DataFrame(clustering_metric_rows)

clustering_metrics.to_csv(
    TABLES_ROOT / "05_kmeans_metrics.csv",
    index=False,
)

if SELECTED_K is None:
    SELECTED_K = int(
        clustering_metrics.loc[
            clustering_metrics["silhouette"].idxmax(),
            "k",
        ]
    )

print("Provisional selected k:", SELECTED_K)
display(clustering_metrics)


In [ ]:
plt.figure(figsize=(8, 5))
plt.plot(
    clustering_metrics["k"],
    clustering_metrics["inertia"],
    marker="o",
)
plt.xlabel("Number of clusters (k)")
plt.ylabel("K-means inertia")
plt.title("K-means inertia")
plt.grid(alpha=0.3)
plt.tight_layout()
plt.savefig(
    FIGURES_ROOT / "05_kmeans_inertia.png",
    dpi=200,
    bbox_inches="tight",
)
plt.show()


In [ ]:
plt.figure(figsize=(8, 5))
plt.plot(
    clustering_metrics["k"],
    clustering_metrics["silhouette"],
    marker="o",
)
plt.xlabel("Number of clusters (k)")
plt.ylabel("Silhouette score")
plt.title("K-means silhouette score")
plt.grid(alpha=0.3)
plt.tight_layout()
plt.savefig(
    FIGURES_ROOT / "05_kmeans_silhouette.png",
    dpi=200,
    bbox_inches="tight",
)
plt.show()


In [ ]:
plt.figure(figsize=(8, 5))
plt.plot(
    clustering_metrics["k"],
    clustering_metrics["davies_bouldin"],
    marker="o",
)
plt.xlabel("Number of clusters (k)")
plt.ylabel("Davies-Bouldin score")
plt.title("K-means Davies-Bouldin score")
plt.grid(alpha=0.3)
plt.tight_layout()
plt.savefig(
    FIGURES_ROOT / "05_kmeans_davies_bouldin.png",
    dpi=200,
    bbox_inches="tight",
)
plt.show()


In [ ]:
selected_labels = clusterings_by_k[SELECTED_K]

cluster_table = manifest[
    ["crystal_id", "sample", "relative_path"]
].copy()

cluster_table["cluster"] = selected_labels

cluster_table.to_csv(
    TABLES_ROOT / f"05_cluster_assignments_k{SELECTED_K}.csv",
    index=False,
)

plt.figure(figsize=(9, 7))
scatter = plt.scatter(
    umap_scores[:, 0],
    umap_scores[:, 1],
    c=selected_labels,
    s=6,
    alpha=0.55,
)
plt.xlabel("UMAP1")
plt.ylabel("UMAP2")
plt.title(f"UMAP coloured by k-means clusters (k={SELECTED_K})")
plt.tight_layout()
plt.savefig(
    FIGURES_ROOT / f"05_umap_by_cluster_k{SELECTED_K}.png",
    dpi=200,
    bbox_inches="tight",
)
plt.show()


# 6. cluster stability

Two tests are used:

1. repeated k-means initialisation with different random seeds;
2. repeated fitting to random 80% subsets, followed by prediction for the full dataset.

Adjusted Rand Index (ARI) near 1 means highly similar partitions. ARI near 0 means agreement similar to chance.


In [ ]:
seed_label_runs = []

for run_seed in tqdm(
    range(SEED, SEED + STABILITY_RUNS),
    desc="K-means seed stability",
):
    labels = KMeans(
        n_clusters=SELECTED_K,
        n_init=20,
        random_state=run_seed,
    ).fit_predict(cluster_features)

    seed_label_runs.append(labels)

seed_ari_matrix = np.eye(STABILITY_RUNS)

for row in range(STABILITY_RUNS):
    for column in range(row + 1, STABILITY_RUNS):
        ari = adjusted_rand_score(
            seed_label_runs[row],
            seed_label_runs[column],
        )
        seed_ari_matrix[row, column] = ari
        seed_ari_matrix[column, row] = ari

upper_triangle = seed_ari_matrix[
    np.triu_indices(STABILITY_RUNS, k=1)
]

seed_stability_summary = pd.DataFrame(
    {
        "metric": [
            "mean_pairwise_ARI",
            "minimum_pairwise_ARI",
            "maximum_pairwise_ARI",
        ],
        "value": [
            upper_triangle.mean(),
            upper_triangle.min(),
            upper_triangle.max(),
        ],
    }
)

seed_stability_summary.to_csv(
    TABLES_ROOT / f"06_seed_stability_k{SELECTED_K}.csv",
    index=False,
)

display(seed_stability_summary)


In [ ]:
plt.figure(figsize=(7, 6))
plt.imshow(seed_ari_matrix, vmin=0, vmax=1)
plt.colorbar(label="Adjusted Rand Index")
plt.xlabel("Run")
plt.ylabel("Run")
plt.title(f"K-means seed stability (k={SELECTED_K})")
plt.tight_layout()
plt.savefig(
    FIGURES_ROOT / f"06_seed_stability_matrix_k{SELECTED_K}.png",
    dpi=200,
    bbox_inches="tight",
)
plt.show()


In [ ]:
reference_labels = seed_label_runs[0]
bootstrap_rows = []
rng = np.random.default_rng(SEED)

for bootstrap_run in tqdm(
    range(STABILITY_RUNS),
    desc="Subset stability",
):
    subset_indices = rng.choice(
        len(cluster_features),
        size=int(0.80 * len(cluster_features)),
        replace=False,
    )

    kmeans = KMeans(
        n_clusters=SELECTED_K,
        n_init=20,
        random_state=SEED + bootstrap_run,
    )

    kmeans.fit(cluster_features[subset_indices])
    predicted_all = kmeans.predict(cluster_features)

    bootstrap_rows.append(
        {
            "run": bootstrap_run,
            "ARI_against_reference": adjusted_rand_score(
                reference_labels,
                predicted_all,
            ),
        }
    )

bootstrap_stability = pd.DataFrame(bootstrap_rows)

bootstrap_stability.to_csv(
    TABLES_ROOT / f"06_subset_stability_k{SELECTED_K}.csv",
    index=False,
)

display(bootstrap_stability.describe())


In [ ]:
plt.figure(figsize=(8, 5))
plt.bar(
    bootstrap_stability["run"],
    bootstrap_stability["ARI_against_reference"],
)
plt.ylim(0, 1)
plt.xlabel("Subset run")
plt.ylabel("ARI against reference clustering")
plt.title(f"Clustering stability under 80% data subsets (k={SELECTED_K})")
plt.tight_layout()
plt.savefig(
    FIGURES_ROOT / f"06_subset_stability_k{SELECTED_K}.png",
    dpi=200,
    bbox_inches="tight",
)
plt.show()


# 7. association with sample identity

Strong sample association may represent genuine geological differences, but it could also reflect scan, preparation, colour-balance, or segmentation differences.

This reports cluster/sample association and how well sample identity can be decoded linearly from the PCA representation.


In [ ]:
sample_labels = manifest["sample"].astype(str).to_numpy()

contingency = pd.crosstab(
    pd.Series(sample_labels, name="sample"),
    pd.Series(selected_labels, name="cluster"),
)

chi2, chi2_p_value, _, _ = chi2_contingency(contingency)
n_observations = contingency.to_numpy().sum()
minimum_dimension = min(
    contingency.shape[0] - 1,
    contingency.shape[1] - 1,
)

cramers_v = math.sqrt(
    chi2
    / np.clip(
        n_observations * minimum_dimension,
        1e-12,
        None,
    )
)

sample_association_metrics = pd.DataFrame(
    {
        "metric": [
            "adjusted_rand_index",
            "adjusted_mutual_information",
            "normalised_mutual_information",
            "homogeneity",
            "completeness",
            "cramers_v",
            "chi_square_p_value",
        ],
        "value": [
            adjusted_rand_score(sample_labels, selected_labels),
            adjusted_mutual_info_score(sample_labels, selected_labels),
            normalized_mutual_info_score(sample_labels, selected_labels),
            homogeneity_score(sample_labels, selected_labels),
            completeness_score(sample_labels, selected_labels),
            cramers_v,
            chi2_p_value,
        ],
    }
)

contingency.to_csv(
    TABLES_ROOT / f"07_sample_cluster_contingency_k{SELECTED_K}.csv"
)

sample_association_metrics.to_csv(
    TABLES_ROOT / f"07_sample_association_metrics_k{SELECTED_K}.csv",
    index=False,
)

display(sample_association_metrics)
display(contingency)


In [ ]:
plt.figure(figsize=(9, 6))
plt.imshow(contingency.to_numpy(), aspect="auto")
plt.colorbar(label="Number of crystals")
plt.xticks(
    np.arange(contingency.shape[1]),
    contingency.columns,
)
plt.yticks(
    np.arange(contingency.shape[0]),
    contingency.index,
)
plt.xlabel("Cluster")
plt.ylabel("Sample")
plt.title(f"Sample-by-cluster counts (k={SELECTED_K})")
plt.tight_layout()
plt.savefig(
    FIGURES_ROOT / f"07_sample_cluster_counts_k{SELECTED_K}.png",
    dpi=200,
    bbox_inches="tight",
)
plt.show()


In [ ]:
sample_classifier = make_pipeline(
    StandardScaler(),
    LogisticRegression(
        max_iter=3_000,
        class_weight="balanced",
    ),
)

sample_cv = StratifiedKFold(
    n_splits=5,
    shuffle=True,
    random_state=SEED,
)

sample_accuracy_scores = cross_val_score(
    sample_classifier,
    pca_scores,
    sample_labels,
    cv=sample_cv,
    scoring="accuracy",
    n_jobs=-1,
)

sample_balanced_accuracy_scores = cross_val_score(
    sample_classifier,
    pca_scores,
    sample_labels,
    cv=sample_cv,
    scoring="balanced_accuracy",
    n_jobs=-1,
)

sample_decoding_summary = pd.DataFrame(
    {
        "metric": [
            "mean_accuracy",
            "std_accuracy",
            "mean_balanced_accuracy",
            "std_balanced_accuracy",
            "chance_accuracy_1_over_8",
        ],
        "value": [
            sample_accuracy_scores.mean(),
            sample_accuracy_scores.std(),
            sample_balanced_accuracy_scores.mean(),
            sample_balanced_accuracy_scores.std(),
            1 / manifest["sample"].nunique(),
        ],
    }
)

sample_decoding_summary.to_csv(
    TABLES_ROOT / "07_sample_decoding_logistic_regression.csv",
    index=False,
)

display(sample_decoding_summary)


# 8. image confounder analysis

computes and caches:

- width and height;
- elongation;
- square-padding fraction used by the model;
- mean and standard deviation of luminance;
- mean red, green, and blue values;
- fraction of near-black pixels.

Then tests:

- how strongly each property correlates with the first ten PCs;
- how well each property can be linearly decoded from the full 50-PC representation;
- how much of each property's variance is associated with cluster membership.

it seems high decodability is not automatically bad and of course crystal morphology and brightness can be geological. The important question is whether they overwhelm the internal geological textures we are trying to pull out from the images with SSL.


In [ ]:
CONFOUNDER_CACHE = TABLES_ROOT / "08_image_confounders.parquet"

if CONFOUNDER_CACHE.exists():
    confounders = pd.read_parquet(CONFOUNDER_CACHE)
    print("Loaded cached confounders:", CONFOUNDER_CACHE)
else:
    confounder_rows = []

    for row in tqdm(
        manifest.itertuples(index=False),
        total=len(manifest),
        desc="Computing image statistics",
    ):
        image_path = Path(row.absolute_path)

        with Image.open(image_path) as image:
            rgb = np.asarray(
                image.convert("RGB"),
                dtype=np.float32,
            )

        width = rgb.shape[1]
        height = rgb.shape[0]
        maximum_dimension = max(width, height)
        minimum_dimension = min(width, height)

        luminance = (
            0.2126 * rgb[:, :, 0]
            + 0.7152 * rgb[:, :, 1]
            + 0.0722 * rgb[:, :, 2]
        )

        near_black = np.max(rgb, axis=2) <= 5

        confounder_rows.append(
            {
                "relative_path": row.relative_path,
                "width_px": width,
                "height_px": height,
                "bounding_box_area_px": width * height,
                "elongation_ratio": (
                    maximum_dimension / minimum_dimension
                ),
                "padding_fraction": (
                    1
                    - (width * height)
                    / (maximum_dimension * maximum_dimension)
                ),
                "mean_luminance": float(luminance.mean()),
                "std_luminance": float(luminance.std()),
                "mean_red": float(rgb[:, :, 0].mean()),
                "mean_green": float(rgb[:, :, 1].mean()),
                "mean_blue": float(rgb[:, :, 2].mean()),
                "near_black_fraction": float(near_black.mean()),
            }
        )

    confounders = pd.DataFrame(confounder_rows)
    confounders.to_parquet(
        CONFOUNDER_CACHE,
        index=False,
    )

display(confounders.head())


In [ ]:
analysis_table = (
    manifest[
        ["crystal_id", "sample", "relative_path"]
    ]
    .merge(
        confounders,
        on="relative_path",
        how="left",
        validate="one_to_one",
    )
)

analysis_table["cluster"] = selected_labels

if analysis_table.isna().any().any():
    raise ValueError("Missing values appeared after merging confounders.")

numeric_confounders = [
    "width_px",
    "height_px",
    "bounding_box_area_px",
    "elongation_ratio",
    "padding_fraction",
    "mean_luminance",
    "std_luminance",
    "mean_red",
    "mean_green",
    "mean_blue",
    "near_black_fraction",
]


In [ ]:
def eta_squared_by_cluster(
    values: np.ndarray,
    labels: np.ndarray,
) -> float:
    values = np.asarray(values, dtype=float)
    grand_mean = values.mean()

    total_sum_squares = np.sum(
        (values - grand_mean) ** 2
    )

    between_sum_squares = 0.0

    for label in np.unique(labels):
        group = values[labels == label]
        between_sum_squares += (
            len(group)
            * (group.mean() - grand_mean) ** 2
        )

    return float(
        between_sum_squares
        / np.clip(total_sum_squares, 1e-12, None)
    )


confounder_metric_rows = []

confounder_cv = KFold(
    n_splits=5,
    shuffle=True,
    random_state=SEED,
)

for column in tqdm(
    numeric_confounders,
    desc="Testing confounders",
):
    target = analysis_table[column].to_numpy(dtype=float)

    correlations = [
        spearmanr(
            target,
            pca_scores[:, component],
        ).statistic
        for component in range(min(10, PCA_COMPONENTS))
    ]

    ridge_model = make_pipeline(
        StandardScaler(),
        Ridge(alpha=1.0),
    )

    r2_scores = cross_val_score(
        ridge_model,
        pca_scores,
        target,
        cv=confounder_cv,
        scoring="r2",
        n_jobs=-1,
    )

    confounder_metric_rows.append(
        {
            "confounder": column,
            "maximum_absolute_spearman_PC1_to_PC10": float(
                np.nanmax(np.abs(correlations))
            ),
            "mean_cross_validated_R2": float(
                r2_scores.mean()
            ),
            "std_cross_validated_R2": float(
                r2_scores.std()
            ),
            "cluster_eta_squared": eta_squared_by_cluster(
                target,
                selected_labels,
            ),
        }
    )

confounder_metrics = (
    pd.DataFrame(confounder_metric_rows)
    .sort_values(
        "mean_cross_validated_R2",
        ascending=False,
    )
)

confounder_metrics.to_csv(
    TABLES_ROOT / f"08_confounder_metrics_k{SELECTED_K}.csv",
    index=False,
)

display(confounder_metrics)


In [ ]:
plt.figure(figsize=(10, 6))
ordered = confounder_metrics.sort_values(
    "mean_cross_validated_R2"
)
plt.barh(
    ordered["confounder"],
    ordered["mean_cross_validated_R2"],
)
plt.xlabel("Mean cross-validated R-squared")
plt.ylabel("Image property")
plt.title("Linear decodability of image properties from SimCLR embeddings")
plt.tight_layout()
plt.savefig(
    FIGURES_ROOT / "08_confounder_linear_decodability.png",
    dpi=200,
    bbox_inches="tight",
)
plt.show()


In [ ]:
plt.figure(figsize=(10, 6))
ordered = confounder_metrics.sort_values(
    "cluster_eta_squared"
)
plt.barh(
    ordered["confounder"],
    ordered["cluster_eta_squared"],
)
plt.xlabel("Cluster eta-squared")
plt.ylabel("Image property")
plt.title(f"Image-property variance associated with clusters (k={SELECTED_K})")
plt.tight_layout()
plt.savefig(
    FIGURES_ROOT / f"08_confounder_cluster_eta_squared_k{SELECTED_K}.png",
    dpi=200,
    bbox_inches="tight",
)
plt.show()


In [ ]:
cluster_confounder_medians = (
    analysis_table
    .groupby("cluster")[numeric_confounders]
    .median()
)

cluster_confounder_medians.to_csv(
    TABLES_ROOT / f"08_cluster_confounder_medians_k{SELECTED_K}.csv"
)

display(cluster_confounder_medians)


# 9. compare SimCLR with SimSiam


The code normalises path separators, reduces each filename to sample/filename, aligns the common crystals, and then compares:

- effective rank;
- linear sample decoding;
- silhouette at the same `k`;
- nearest-neighbour overlap;
- agreement between the two models' k-means partitions.


In [ ]:
def normalise_plagnet_relative_path(value: str) -> str:
    text = str(value).replace("\\", "/")
    parts = [part for part in text.split("/") if part]

    expected_samples = set(
        manifest["sample"].astype(str).unique()
    )

    for position, part in enumerate(parts):
        if part in expected_samples and position + 1 < len(parts):
            return f"{part}/{parts[-1]}"

    if len(parts) >= 2:
        return "/".join(parts[-2:])

    return parts[-1]


def load_legacy_simsiam(path: Path):
    with np.load(path, allow_pickle=True) as saved:
        print("SimSiam arrays:", saved.files)
        values = np.asarray(
            saved["embeddings"],
            dtype=np.float32,
        )
        filenames = np.asarray(
            saved["filenames"],
            dtype=str,
        )

    relative_paths = np.array(
        [
            normalise_plagnet_relative_path(value)
            for value in filenames
        ],
        dtype=str,
    )

    return values, relative_paths


def representation_summary(
    values: np.ndarray,
    labels: np.ndarray,
    sample_names_for_values: np.ndarray,
    model_name: str,
) -> dict:
    values_l2 = values / np.clip(
        np.linalg.norm(values, axis=1, keepdims=True),
        1e-12,
        None,
    )

    pca = PCA(
        n_components=min(PCA_COMPONENTS, values.shape[1]),
        svd_solver="randomized",
        random_state=SEED,
    )
    scores = pca.fit_transform(values_l2)
    cluster_input = Normalizer().fit_transform(scores)

    clusters = KMeans(
        n_clusters=SELECTED_K,
        n_init=20,
        random_state=SEED,
    ).fit_predict(cluster_input)

    classifier = make_pipeline(
        StandardScaler(),
        LogisticRegression(
            max_iter=3_000,
            class_weight="balanced",
        ),
    )

    cv = StratifiedKFold(
        n_splits=5,
        shuffle=True,
        random_state=SEED,
    )

    balanced_scores = cross_val_score(
        classifier,
        scores,
        sample_names_for_values,
        cv=cv,
        scoring="balanced_accuracy",
        n_jobs=-1,
    )

    effective_rank, _ = effective_rank_from_embeddings(values)

    return {
        "model": model_name,
        "images": len(values),
        "dimensions": values.shape[1],
        "effective_rank": effective_rank,
        "silhouette_at_selected_k": silhouette_score(
            cluster_input,
            clusters,
            sample_size=min(SILHOUETTE_SAMPLE_SIZE, len(values)),
            random_state=SEED,
        ),
        "sample_balanced_accuracy": balanced_scores.mean(),
        "sample_balanced_accuracy_std": balanced_scores.std(),
        "clusters": clusters,
        "normalised_embeddings": values_l2,
    }


In [ ]:
if SIMSIAM_NPZ is None:
    print(
        "Step 9 is ready but skipped. Set SIMSIAM_NPZ to the "
        "legacy SSL_output_20260408_135606.npz file and rerun this cell." # just check thi as everything will have been repackaged recently
    )
else:
    SIMSIAM_NPZ = Path(SIMSIAM_NPZ)

    if not SIMSIAM_NPZ.exists():
        raise FileNotFoundError(SIMSIAM_NPZ)

    simsiam_embeddings, simsiam_paths = load_legacy_simsiam(
        SIMSIAM_NPZ
    )

    simclr_lookup = {
        path: index
        for index, path in enumerate(manifest_paths)
    }

    simsiam_lookup = {
        path: index
        for index, path in enumerate(simsiam_paths)
    }

    common_paths = sorted(
        set(simclr_lookup)
        & set(simsiam_lookup)
    )

    print("Common aligned crystals:", len(common_paths))

    if len(common_paths) != 12_344:
        print(
            "WARNING: the models do not contain exactly the same "
            "12,344 image paths. The comparison will use only common paths."
        )

    simclr_indices = np.array(
        [simclr_lookup[path] for path in common_paths]
    )

    simsiam_indices = np.array(
        [simsiam_lookup[path] for path in common_paths]
    )

    aligned_simclr = embeddings_raw[simclr_indices]
    aligned_simsiam = simsiam_embeddings[simsiam_indices]

    aligned_samples = manifest.iloc[
        simclr_indices
    ]["sample"].astype(str).to_numpy()

    simclr_result = representation_summary(
        aligned_simclr,
        selected_labels[simclr_indices],
        aligned_samples,
        "SimCLR",
    )

    simsiam_result = representation_summary(
        aligned_simsiam,
        np.zeros(len(aligned_simsiam)),
        aligned_samples,
        "SimSiam",
    )

    # Nearest-neighbour overlap at the same top-k.
    simclr_nn = NearestNeighbors(
        n_neighbors=NEIGHBOUR_COUNT + 1,
        metric="cosine",
        n_jobs=-1,
    ).fit(
        simclr_result["normalised_embeddings"]
    ).kneighbors(
        return_distance=False
    )[:, 1:]

    simsiam_nn = NearestNeighbors(
        n_neighbors=NEIGHBOUR_COUNT + 1,
        metric="cosine",
        n_jobs=-1,
    ).fit(
        simsiam_result["normalised_embeddings"]
    ).kneighbors(
        return_distance=False
    )[:, 1:]

    neighbour_overlaps = []

    for row in range(len(common_paths)):
        overlap = len(
            set(simclr_nn[row])
            & set(simsiam_nn[row])
        ) / NEIGHBOUR_COUNT
        neighbour_overlaps.append(overlap)

    clustering_agreement = adjusted_rand_score(
        simclr_result["clusters"],
        simsiam_result["clusters"],
    )

    comparison = pd.DataFrame(
        [
            {
                key: value
                for key, value in simclr_result.items()
                if key not in {
                    "clusters",
                    "normalised_embeddings",
                }
            },
            {
                key: value
                for key, value in simsiam_result.items()
                if key not in {
                    "clusters",
                    "normalised_embeddings",
                }
            },
        ]
    )

    comparison["mean_top10_neighbour_overlap"] = (
        np.mean(neighbour_overlaps)
    )

    comparison["cross_model_cluster_ARI"] = (
        clustering_agreement
    )

    comparison.to_csv(
        TABLES_ROOT / "09_simclr_simsiam_comparison.csv",
        index=False,
    )

    pd.DataFrame(
        {
            "relative_path": common_paths,
            "top10_neighbour_overlap": neighbour_overlaps,
        }
    ).to_csv(
        TABLES_ROOT / "09_per_crystal_neighbour_overlap.csv",
        index=False,
    )

    display(comparison)


# final analysis record

writes compact record of main settings and generated files.


In [ ]:
analysis_record = {
    "simclr_run": str(SIMCLR_RUN),
    "embedding_file": str(EMBEDDING_FILE),
    "number_of_images": int(len(manifest)),
    "embedding_dimensions": int(embeddings_raw.shape[1]),
    "seed": SEED,
    "pca_components": PCA_COMPONENTS,
    "umap_n_neighbors": UMAP_NEIGHBOURS,
    "umap_min_dist": UMAP_MIN_DIST,
    "tested_k_values": K_VALUES,
    "selected_k": int(SELECTED_K),
    "nearest_neighbour_count": NEIGHBOUR_COUNT,
    "mean_top10_same_sample_rate": float(
        same_sample_neighbour_rate
    ),
    "effective_rank": float(effective_rank),
    "analysis_completed": True,
}

with (ANALYSIS_ROOT / "analysis_record.json").open("w") as handle:
    json.dump(
        analysis_record,
        handle,
        indent=2,
    )

print(json.dumps(analysis_record, indent=2))
print("\nAnalysis outputs saved under:")
print(ANALYSIS_ROOT)
